# Feed workbench (development only)

Try a vendor feed's mapping and rules against a sample file **without loading anything**.

* Runs the *same* `lbp` code as the production Glue jobs (`lbp-feed`, `lbp-feed-spark`).
* Reads the feed's **published, signed configuration** (what production runs) or, optionally, a draft mapping version for comparison.
* Production never runs notebooks: changes are made in the web app, approved by an SME, and published. This notebook only reads.

In [ ]:
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2
%idle_timeout 30
%connections lbp-vpc
%additional_python_modules s3://REPLACE-artifacts-bucket/wheels/lbp-1.0.0-py3-none-any.whl
%%configure
{"--LBP_ENV": "aws", "--LBP_DB_SECRET_ARN": "REPLACE", "--LBP_DB_HOST": "REPLACE", "--LBP_PROCESSED_BUCKET": "REPLACE",
 "--LBP_CONFIG_SIGNING_SECRET_ARN": "REPLACE"}

In [ ]:
import os, sys
from awsglue.utils import getResolvedOptions
# Session parameters become LBP_* settings for the lbp package (same settings the jobs use)
for k in ["LBP_ENV", "LBP_DB_SECRET_ARN", "LBP_DB_HOST", "LBP_PROCESSED_BUCKET", "LBP_CONFIG_SIGNING_SECRET_ARN"]:
    try:
        os.environ[k] = getResolvedOptions(sys.argv, [k])[k]
    except Exception:
        pass
os.environ["LBP_AUTO_CREATE_SCHEMA"] = "false"

## 1. Pick the feed and a sample file

In [ ]:
FEED_ID = "src_REPLACE"                                   # Vendors & Mapping -> feed -> id
SAMPLE = "s3://REPLACE-raw/incoming/src_REPLACE/sample.csv"  # any file this feed sends
DRAFT_MAPPING_ID = None                                    # optional: compare a draft version before approving it

In [ ]:
from lbp import config_store
cfg = config_store.load(FEED_ID)   # verifies the signature, exactly as the job does
print(f"{cfg.feed_name}: published v{cfg.version} (mapping v{cfg.mapping_version}), {len(cfg.fields)} fields,",
      {k: len(v) for k, v in cfg.runtime_set.items()}, "rules")

## 2. Run the published configuration over the sample

In [ ]:
import pandas as pd
from lbp.ingestion.parser import ParseOptions, read_sample
from lbp.pipeline.batch import process_chunk
from lbp.storage import local_path

meta, rows = read_sample(local_path(SAMPLE), 2000, ParseOptions(**cfg.parse_options))
out = process_chunk(rows, cfg.fields, cfg.runtime_set, 0, f"nb:{cfg.mapping_id}:{cfg.mapping_version}|{cfg.rules_key}")
summary = pd.Series([o["status"] for o in out]).value_counts()
fails = pd.DataFrame([{"rule": f["rule_id"], "field": f["field"], "severity": f["severity"]} for o in out for f in o["fails"]])
display(summary)
display(fails.groupby(["rule", "field", "severity"]).size().sort_values(ascending=False).head(20).to_frame("rows"))

## 3. Transform errors by field (mapping problems, not data problems)

In [ ]:
terrs = pd.DataFrame([{"field": k, "error": v.split(":")[0]} for o in out for k, v in o["terrs"].items()])
display(terrs.value_counts().head(20).to_frame("rows") if len(terrs) else "no transform errors")

## 4. Optional: compare a draft mapping version

In [ ]:
if DRAFT_MAPPING_ID:
    from lbp.mapping.service import get_mapping
    draft = get_mapping(DRAFT_MAPPING_ID)
    out2 = process_chunk(rows, draft["fields"], cfg.runtime_set, 0, f"nb-draft:{DRAFT_MAPPING_ID}")
    cmp = pd.DataFrame({"published": pd.Series([o["status"] for o in out]).value_counts(),
                        "draft": pd.Series([o["status"] for o in out2]).value_counts()}).fillna(0).astype(int)
    display(cmp)

## 5. At scale: the Spark path on the full file

In [ ]:
from lbp import glue_jobs
proc = glue_jobs.partition_processor(cfg.fields, cfg.runtime_set, f"{cfg.mapping_id}:{cfg.mapping_version}|{cfg.rules_key}")
df = spark.read.option("header", True).option("sep", cfg.parse_options.get("delimiter") or ",").csv(SAMPLE)
def run(pdf_iter):
    import pandas as pd
    for pdf in pdf_iter:
        res = proc(pdf.fillna("").astype(str).to_dict("records"), 0)
        yield pd.DataFrame({"status": [r["status"] for r in res]})
df.mapInPandas(run, "status string").groupBy("status").count().show()